In [ ]:
import json
import pandas as pd

In [ ]:
with open('/content/map.txt') as f:
  data = json.load(f)

# Нормализуем данные в плоский DataFrame
map = data["Карта"]
prob = data["Вероятности нападения"]
rel = data["Связи между засадами"]

Посмотрю на цепочки уникальных маршрутов

In [ ]:
tracks = [['Басра']]
op_cnt=1
while op_cnt>0:
  op_cnt=0
  tracks2=[]
  for cur_tr in tracks:
    cur_ct = cur_tr[-1]
    tmp_l = []
    for ch in map:
      if ch[0]==cur_ct:
        tmp_l.append(ch[1])
        op_cnt+=1
    for y in tmp_l:
      tracks2.append(cur_tr+[y])
  if op_cnt>0:
    tracks = tracks2
#
print(len(tracks))

9


Просто посмотрю, есть ли вероятность путешествия без нападений

In [ ]:
max_p0 = -1
opt_track = -1
for tr in tracks:
  p0=1
  for ct in tr:
    p0*=(1-prob.get(ct,0))
  if p0>max_p0:
    max_p0 = p0
    opt_track = tr
  #max_p0 = max(max_p0,p0)
print(max_p0)
print(opt_track)

0.009689043419999998
['Басра', 'Аль-Кадисия', 'Джабаль-Нур', 'Аль-Фаллуджа', 'Джебель-Друз', 'Аль-Мафрак', 'Джабаль-Умм', 'Аль-Хамадия', 'Дамаск']


С учетом того, что караван может выдержать до 3 нападений

In [ ]:
max_prob = -1
opt_track = -1
rob_set = -1
for tr in tracks:
  for i1,rob1_ct in enumerate(tr):
    for i2,rob2_ct in enumerate(tr):
      for i3,rob3_ct in enumerate(tr):
        if i1<i2<i3:
          p3 = 1
          for j,ct in enumerate(tr):
            if j==i1 or j==i2 or j==i3:
              p3*=1
            else:
              s_key1 = "('" + rob1_ct + "', '" + ct + "')"
              s_key2 = "('" + rob2_ct + "', '" + ct + "')"
              s_key3 = "('" + rob3_ct + "', '" + ct + "')"
              add_p = 0
              if s_key1 in rel:
                add_p += rel[s_key1]
              if s_key2 in rel:
                add_p += rel[s_key2]
              if s_key3 in rel:
                add_p += rel[s_key3]

              p3 *= (1-prob.get(ct,0)-add_p)
          #max_p3 = max(max_p3,p3)
          if p3>max_prob:
            max_prob = p3
            opt_track = tr
            rob_set = (rob1_ct,rob2_ct,rob3_ct)
print(max_prob)
print(opt_track)
print(rob_set)

0.13307027999999999
['Басра', 'Аль-Кадисия', 'Джабаль-Нур', 'Аль-Фаллуджа', 'Аль-Фарафра', 'Аль-Кунейтра', 'Вади-Муса', 'Аль-Кусейма', 'Дамаск']
('Аль-Кадисия', 'Аль-Кунейтра', 'Вади-Муса')


Максимальная вероятность успешного путешествия 0.13